# Phase 12 — Observability and Monitoring

**What you will learn**
- How logging works in Kubernetes: stdout/stderr, node log files, `kubectl logs`, log agents.
- Metrics: `metrics-server` vs **Prometheus**, and when to use each.
- Install **kube-prometheus-stack** (Prometheus, Alertmanager, Grafana, exporters) on kind.
- **PromQL** basics with real queries; scrape your own app with a **ServiceMonitor**.
- **Grafana** dashboards and building a panel.
- **Alertmanager** and **PrometheusRule** alerts (CrashLooping pod, high memory, error rate).
- **Loki** for logs and **LogQL** queries; shipping logs with **Grafana Alloy**.
- **OpenTelemetry** and **distributed tracing** basics.
- RED, USE and the four golden signals; alerting strategy; production dashboards.
- How all of this maps to your production **Prometheus + Thanos** setup.

**Time:** 5–6 hours.

**Prerequisites**
- Notebooks 00–11. Helm installed (notebook 09).
- Docker Desktop memory: **at least 6 GB, 8 GB recommended**. The monitoring stack + Loki use about 2–3 GB.

**How to use this notebook**
- Run cells top to bottom with **Shift+Enter**. Read the text before each cell.
- Some cells start a `port-forward` in the background. The cleanup cell stops them.
- Quiz answers are hidden under **Answer**. Try first.

## 0. Setup

In [ ]:
%%bash
# GUARD: stop if we are not on the local lab cluster.
ctx=$(kubectl config current-context 2>/dev/null)
if [ "$ctx" != "kind-learn" ]; then
  echo "STOP: current context is '$ctx', not 'kind-learn'."
  echo "Fix:  kubectl config use-context kind-learn"
  exit 1
fi
echo "OK: using kind-learn"

We use two namespaces:
- `lab-12` for our sample apps.
- `monitoring` for the monitoring tools (shared; notebook 13 can use it too).

In [ ]:
%%bash
mkdir -p labs/12
kubectl create namespace lab-12 --dry-run=client -o yaml | kubectl apply -f -
kubectl create namespace monitoring --dry-run=client -o yaml | kubectl apply -f -
helm repo add prometheus-community https://prometheus-community.github.io/helm-charts >/dev/null
helm repo add grafana https://grafana.github.io/helm-charts >/dev/null
helm repo add open-telemetry https://open-telemetry.github.io/opentelemetry-helm-charts >/dev/null
helm repo add metrics-server https://kubernetes-sigs.github.io/metrics-server/ >/dev/null
helm repo update >/dev/null && echo "helm repos ready"

**Observability** = being able to understand what is happening inside a system **from the outside**, using the data it produces. The three main data types ("three pillars"):

| Pillar | Question it answers | Example | Tool in this notebook |
|---|---|---|---|
| **Logs** | What exactly happened? | `ERROR payment timeout order=123` | Loki |
| **Metrics** | How much / how often / how fast? Is it getting worse? | `http_requests_total`, CPU % | Prometheus |
| **Traces** | Where did the time go across services? | checkout → payment (800 ms) → bank API | OpenTelemetry + Tempo/Jaeger |

---
## 1. Logging in Kubernetes

### What is it?
In Kubernetes, an app should write logs to **stdout** (normal output) and **stderr** (error output) — not to files inside the container. The container runtime saves this output to files on the **node**.

### Why?
- Containers are temporary. If logs are inside the container, they disappear when the pod is deleted.
- One standard place on every node lets one log agent collect everything.

### How it works internally

```
 app process ──stdout/stderr──▶ containerd ──▶ /var/log/pods/<ns>_<pod>_<uid>/<container>/0.log
                                                   ▲  (symlinks in /var/log/containers/)
 kubectl logs ──▶ API server ──▶ kubelet (on that node) reads the file
 log agent (DaemonSet on every node) ──reads files / API──▶ Loki / OpenSearch / OCI Logging
```

Important facts:
- The kubelet **rotates** these files (default 10 MB, 5 files). Old logs are lost from the node.
- When a pod is **deleted**, its log files are deleted too. Only a central log system keeps history.
- `kubectl logs --previous` shows the log of the **last crashed** container instance — very useful for CrashLoopBackOff.

### Real production scenario
A payment pod crashes at 03:00 and restarts. At 09:00 the team runs `kubectl logs` and sees only the clean startup of the new container. The error is gone. With `--previous` they see the last crash only. With a central log system (Loki) they search all crashes since 03:00.

### Lab: logs from stdout and stderr

In [ ]:
%%writefile labs/12/logger.yaml
apiVersion: v1
kind: Pod
metadata:
  name: logger
  namespace: lab-12
  labels:
    app: logger
spec:
  containers:
    - name: app
      image: busybox:1.36
      # Write one INFO line to stdout and one ERROR line to stderr every 3 seconds.
      command: ["sh", "-c"]
      args:
        - |
          i=0
          while true; do
            i=$((i+1))
            echo "$(date -u +%FT%TZ) level=info msg=\"order processed\" order=$i"
            if [ $((i % 5)) -eq 0 ]; then
              echo "$(date -u +%FT%TZ) level=error msg=\"payment timeout\" order=$i" >&2
            fi
            sleep 3
          done

In [ ]:
%%bash
kubectl apply -f labs/12/logger.yaml
kubectl -n lab-12 wait --for=condition=Ready pod/logger --timeout=90s
sleep 16
echo "--- last 5 lines ---"
kubectl -n lab-12 logs logger --tail=5
echo "--- only last 10 seconds, with timestamps added by Kubernetes ---"
kubectl -n lab-12 logs logger --since=10s --timestamps

Useful `kubectl logs` options:

| Command | What it does |
|---|---|
| `kubectl logs POD` | Log of the (only) container |
| `kubectl logs POD -c NAME` | Choose a container in a multi-container pod |
| `kubectl logs POD --previous` | Log of the previous (crashed) container instance |
| `kubectl logs -f POD` | Follow (stream) new lines |
| `kubectl logs -l app=logger --all-containers` | All pods with a label |
| `kubectl logs deploy/NAME` | One pod of a Deployment |
| `--since=1h`, `--tail=100`, `--timestamps` | Filter and add time |

Now look at the **real files on the node**. kind nodes are Docker containers, so we use `docker exec`.

In [ ]:
%%bash
node=$(kubectl -n lab-12 get pod logger -o jsonpath='{.spec.nodeName}')
echo "logger runs on node: $node"
docker exec "$node" sh -c 'ls -l /var/log/containers/ | grep logger'
docker exec "$node" sh -c 'tail -n 3 /var/log/pods/lab-12_logger_*/app/0.log'

**Expected:** each line in the file has a prefix: `<timestamp> stdout F ...` or `<timestamp> stderr F ...`. This is the **CRI log format**. Log agents parse it.

### Lab: `--previous` for a crashing container

In [ ]:
%%bash
kubectl -n lab-12 run crasher --image=busybox:1.36 --restart=Always -- sh -c 'echo "starting"; echo "FATAL: config file /etc/app.conf missing" >&2; exit 1'
sleep 25
kubectl -n lab-12 get pod crasher
echo "--- current log (may be empty or a fresh start) ---"
kubectl -n lab-12 logs crasher || true
echo "--- previous instance log ---"
kubectl -n lab-12 logs crasher --previous || true
kubectl -n lab-12 delete pod crasher --now

### Log agents (DaemonSet)
A **log agent** runs on every node (as a **DaemonSet**), reads the log files (or the API), adds labels (namespace, pod, container), and sends logs to a backend.

| Agent | Notes |
|---|---|
| **Grafana Alloy** | Grafana's collector (replaces Promtail, which reached end of life in 2026). Logs, metrics, traces. We use it below. |
| **Fluent Bit** | Very light, common in production, many outputs (Loki, OpenSearch, OCI Logging, S3). |
| **OpenTelemetry Collector** | Vendor-neutral; can collect logs with the `filelog` receiver. |
| OCI / cloud agents | OKE can send container logs to OCI Logging via the Unified Monitoring Agent. |

### Troubleshooting logs

| Symptom | Cause | Fix |
|---|---|---|
| `kubectl logs` empty | App writes to a file, not stdout | Change app config to log to stdout, or add a sidecar that tails the file |
| `previous terminated container not found` | Container never restarted | Normal; use current logs |
| Logs stop after some hours on the node | Rotation | Use a central log system |
| `Error from server: ... dial tcp ...:10250` | API server cannot reach kubelet | Node/kubelet problem (see notebook 13) |

### Lab vs production

| Lab | Production |
|---|---|
| `kubectl logs` | Central logs (Loki/OpenSearch/OCI Logging) with retention (e.g. 30–90 days) |
| Free text lines | **Structured logs** (JSON or key=value) with `trace_id`, `request_id` |
| No limits | Log volume budgets; drop debug logs; no secrets or personal data in logs |

### Check yourself
1. Where on the node are container logs stored?
2. What happens to the logs when a pod is deleted?
3. Your pod is in CrashLoopBackOff. Which `kubectl logs` flag shows the error?
4. Why does a log agent run as a DaemonSet?

<details><summary>Answer</summary>

1. `/var/log/pods/<namespace>_<pod>_<uid>/<container>/N.log` (with symlinks in `/var/log/containers/`).
2. They are deleted from the node. Only a central log system keeps them.
3. `--previous`.
4. Logs are files on each node. One agent per node reads that node's files.
</details>

---
## 2. Metrics: metrics-server vs Prometheus

### What is it?
A **metric** is a number measured over time, with labels. Example: `container_cpu_usage_seconds_total{namespace="lab-12", pod="logger"} 12.5`.

| | **metrics-server** | **Prometheus** |
|---|---|---|
| Purpose | Current CPU/memory for `kubectl top`, **HPA**, VPA | Full monitoring: any metric, history, alerts |
| History | None (only the latest value) | Yes (days, or years with Thanos) |
| Data source | kubelet resource metrics | Scrapes `/metrics` HTTP endpoints of anything |
| Query language | None | PromQL |
| Alerts | No | Yes (with Alertmanager) |

You usually run **both**: metrics-server for autoscaling, Prometheus for monitoring.

### Lab: make sure metrics-server works
kind kubelets use self-signed certificates, so metrics-server needs `--kubelet-insecure-tls` (**lab only**; in production use proper kubelet serving certificates). Notebook 07 may have installed it already — this cell installs it only if missing.

In [ ]:
%%bash
if kubectl -n kube-system get deploy metrics-server >/dev/null 2>&1; then
  echo "metrics-server already installed"
else
  helm upgrade --install metrics-server metrics-server/metrics-server -n kube-system \
    --set 'args={--kubelet-insecure-tls}' --wait --timeout 3m
fi
sleep 20
kubectl top nodes
kubectl -n lab-12 top pod || echo "metrics not ready yet - wait 30s and run: kubectl -n lab-12 top pod"

**Expected:** a table of nodes with CPU (cores, `m` = millicores) and memory. If you see `metrics not available yet`, wait 30 seconds and run again.

---
## 3. Prometheus

### What is it?
**Prometheus** is a time-series database and monitoring system. It **pulls** (scrapes) metrics from HTTP endpoints every N seconds, stores them, evaluates alert rules, and answers PromQL queries.

### How it works internally

```
                         ┌────────────── Prometheus ──────────────┐
 targets (/metrics)      │ service discovery (Kubernetes API)     │
  node-exporter  ◀─scrape│ scrape loop every 15-30s               │
  kube-state-metrics ◀───│ TSDB (local disk, blocks of 2h)        │──▶ Grafana (PromQL)
  kubelet/cAdvisor ◀─────│ rule engine: recording + alert rules ──│──▶ Alertmanager ──▶ Slack/Teams/email
  your app ◀─────────────│                                        │──▶ remote_write / Thanos sidecar
                         └────────────────────────────────────────┘
```

Key exporters installed by kube-prometheus-stack:
- **node-exporter** (DaemonSet): node CPU, memory, disk, network. → *the disk-full alerts you know*.
- **kube-state-metrics**: the **state of Kubernetes objects** — pod phase, restarts, Deployment replicas, resource requests/limits.
- **kubelet / cAdvisor**: per-container CPU and memory usage.

### The Prometheus Operator
kube-prometheus-stack uses the **Prometheus Operator**. Instead of editing `prometheus.yml`, you create Kubernetes objects:
- `ServiceMonitor` / `PodMonitor` — "scrape these Services/Pods".
- `PrometheusRule` — alert and recording rules.
- `Prometheus`, `Alertmanager` — the servers themselves.
The operator turns these objects into config and reloads Prometheus.

### Mapping to your Thanos setup

| Your production | Here |
|---|---|
| 2 Prometheus replicas + Thanos sidecar | 1 Prometheus (lab) |
| Thanos Query (global view, dedup of replicas) | Query Prometheus directly |
| Thanos Store + Compactor, OCI Object Storage, 2-year retention | Local disk, `retention: 1d` |
| OKE clusters `remote_write` to the hub | Same cluster |
| Grafana on the hub | Grafana in `monitoring` namespace |

PromQL, ServiceMonitors, PrometheusRules and Alertmanager config work **the same** in both. Thanos adds long retention, global querying and HA dedup.

### Lab: install kube-prometheus-stack (small, lab sizing)

Read the values. We disable scraping of etcd, controller-manager, scheduler and kube-proxy because on kind they listen only on `127.0.0.1` and would show as "down". In production (self-managed clusters) you enable them; on managed clusters like OKE the control plane is managed by Oracle and not scrapeable.

In [ ]:
%%writefile labs/12/kps-values.yaml
fullnameOverride: kps                 # Short names: kps-prometheus, kps-alertmanager, kps-grafana

prometheus:
  prometheusSpec:
    retention: 1d                     # Lab: keep 1 day. Production: 15d local + Thanos for long term
    scrapeInterval: 30s
    resources:
      requests: {cpu: 100m, memory: 400Mi}
      limits: {memory: 1Gi}
    # Pick up ServiceMonitors / PodMonitors / Rules from ANY namespace and with ANY labels.
    # Default = only objects with the Helm release label. Lab convenience.
    serviceMonitorSelectorNilUsesHelmValues: false
    podMonitorSelectorNilUsesHelmValues: false
    ruleSelectorNilUsesHelmValues: false

alertmanager:
  alertmanagerSpec:
    resources:
      requests: {cpu: 10m, memory: 32Mi}

grafana:
  adminPassword: learn-admin          # LAB ONLY. Production: from a Secret / SSO
  persistence:
    enabled: false                    # Lab: dashboards created in UI are lost on restart
  sidecar:
    datasources:
      enabled: true                   # Load data sources from ConfigMaps (we add Loki later)
    dashboards:
      enabled: true                   # Load dashboards from ConfigMaps with label grafana_dashboard=1

# Not reachable on kind (bound to 127.0.0.1 on the control-plane node):
kubeEtcd:              {enabled: false}
kubeControllerManager: {enabled: false}
kubeScheduler:         {enabled: false}
kubeProxy:             {enabled: false}

prometheus-node-exporter:
  hostRootFsMount:
    enabled: false                    # kind on Docker Desktop: avoid mount propagation errors

This installs the stack. `--wait` waits until pods are ready. First install can take 3–6 minutes (image downloads).

In [ ]:
%%bash
helm upgrade --install kps prometheus-community/kube-prometheus-stack \
  -n monitoring -f labs/12/kps-values.yaml --wait --timeout 10m
kubectl -n monitoring get pods

**Expected:** all `Running`:
```
alertmanager-kps-alertmanager-0               2/2   Running
kps-grafana-xxxxx                             3/3   Running
kps-kube-state-metrics-xxxxx                  1/1   Running
kps-operator-xxxxx                            1/1   Running
kps-prometheus-node-exporter-xxxxx (x3)       1/1   Running
prometheus-kps-prometheus-0                   2/2   Running
```

Start port-forwards in the background for Prometheus (9090), Grafana (3000) and Alertmanager (9093).

In [ ]:
%%bash
start_pf() {  # name svc localport:remoteport
  nohup kubectl -n monitoring port-forward svc/$2 $3 > labs/12/pf-$1.log 2>&1 &
  echo $! > labs/12/pf-$1.pid
}
start_pf prom kps-prometheus 9090:9090
start_pf grafana kps-grafana 3000:80
start_pf am kps-alertmanager 9093:9093
sleep 4
curl -s localhost:9090/-/ready; echo
curl -s -o /dev/null -w "Grafana HTTP %{http_code}\n" localhost:3000/login
curl -s -o /dev/null -w "Alertmanager HTTP %{http_code}\n" localhost:9093/-/ready

Open **http://localhost:9090** → *Status → Targets*. All targets should be **UP**.

### PromQL basics
PromQL = Prometheus Query Language. Main ideas:
- **Instant vector:** current value per series. `up`
- **Label filter:** `up{job="kubelet"}`, regex `=~`, not equal `!=`
- **Range vector:** values over time: `[5m]`. Needed for `rate()`.
- **Counter** (only goes up, e.g. `_total`): always use `rate()` or `increase()`.
- **Gauge** (goes up and down, e.g. memory): use directly or with `avg_over_time()`.
- **Aggregation:** `sum by (namespace) (...)`, `topk(5, ...)`.

We run queries through the HTTP API with a small helper, so the notebook shows results. You can paste the same queries in the Prometheus UI.

In [ ]:
%%bash
cat > labs/12/q.sh <<'EOF'
#!/bin/sh
# Usage: sh labs/12/q.sh '<promql>'   -> prints "labels => value" per series
curl -s localhost:9090/api/v1/query --data-urlencode "query=$1" | python3 -c '
import json,sys
d=json.load(sys.stdin)
if d.get("status")!="success": print(d); sys.exit()
for r in d["data"]["result"][:15]:
    m={k:v for k,v in r["metric"].items() if k not in ("__name__","endpoint","service","prometheus","instance","job")} or r["metric"]
    print(m, "=>", r["value"][1])
'
EOF
echo "1) Which targets are up (1) or down (0)?"
sh labs/12/q.sh 'up'
echo; echo "2) CPU cores used per namespace (rate over 5m):"
sh labs/12/q.sh 'sum by (namespace) (rate(container_cpu_usage_seconds_total{container!=""}[5m]))'
echo; echo "3) Top 5 pods by memory working set (MiB):"
sh labs/12/q.sh 'topk(5, sum by (namespace,pod) (container_memory_working_set_bytes{container!=""})) / 1024 / 1024'
echo; echo "4) Container restarts in the last 1h:"
sh labs/12/q.sh 'sum by (namespace,pod) (increase(kube_pod_container_status_restarts_total[1h])) > 0'
echo; echo "5) Node filesystem free %:"
sh labs/12/q.sh '100 * node_filesystem_avail_bytes{mountpoint="/",fstype!="tmpfs"} / node_filesystem_size_bytes{mountpoint="/",fstype!="tmpfs"}'

**What to notice**
- Query 2 uses `rate()` because `container_cpu_usage_seconds_total` is a **counter** (seconds of CPU used, always growing). `rate` gives "seconds per second" = cores.
- Query 3 uses `working_set` — this is the memory number the kernel's OOM killer looks at (close to it). Not `usage`, which includes cache.
- Query 4 comes from **kube-state-metrics**.
- Query 5 is the classic "disk almost full" query from node-exporter.

### Lab: scrape your own app with a ServiceMonitor
We deploy **podinfo**, a small demo web app that exposes Prometheus metrics on port 9797 (`/metrics`) and has test endpoints: `/status/500` (returns an error) and `/delay/1` (slow response).

In [ ]:
%%writefile labs/12/podinfo.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: podinfo
  namespace: lab-12
spec:
  replicas: 2
  selector:
    matchLabels: {app: podinfo}
  template:
    metadata:
      labels: {app: podinfo}
    spec:
      containers:
        - name: podinfo
          image: ghcr.io/stefanprodan/podinfo:6.7.1
          command: ["./podinfo", "--port=9898", "--port-metrics=9797", "--level=info"]
          ports:
            - {name: http, containerPort: 9898}          # app traffic
            - {name: http-metrics, containerPort: 9797}  # Prometheus /metrics
          resources:
            requests: {cpu: 20m, memory: 32Mi}
            limits: {memory: 128Mi}
          readinessProbe:
            httpGet: {path: /readyz, port: 9898}
---
apiVersion: v1
kind: Service
metadata:
  name: podinfo
  namespace: lab-12
  labels: {app: podinfo}             # The ServiceMonitor selects Services by THESE labels
spec:
  selector: {app: podinfo}
  ports:
    - {name: http, port: 9898, targetPort: http}
    - {name: http-metrics, port: 9797, targetPort: http-metrics}   # port NAME used by ServiceMonitor
---
apiVersion: monitoring.coreos.com/v1
kind: ServiceMonitor
metadata:
  name: podinfo
  namespace: lab-12
spec:
  selector:
    matchLabels: {app: podinfo}      # Which Services to scrape
  namespaceSelector:
    matchNames: [lab-12]             # In which namespaces
  endpoints:
    - port: http-metrics             # Service port NAME (not number)
      path: /metrics
      interval: 15s

In [ ]:
%%bash
kubectl apply -f labs/12/podinfo.yaml
kubectl -n lab-12 rollout status deploy/podinfo --timeout=120s
# --command: run 'sh' instead of the image entrypoint (curl)
kubectl -n lab-12 run load --image=curlimages/curl:8.10.1 --restart=Never --command -- sh -c '
  while true; do
    curl -s -o /dev/null http://podinfo:9898/
    curl -s -o /dev/null http://podinfo:9898/status/500
    curl -s -o /dev/null http://podinfo:9898/delay/1 &
    sleep 0.5
  done'
echo "Load generator started. Raw metric names exposed by podinfo:"
kubectl -n lab-12 run peek --rm -i --restart=Never --image=curlimages/curl:8.10.1 -- -s http://podinfo:9797/metrics | grep -E '^# TYPE http_'

Wait about 60 seconds (scrape interval + rate window), then query. If metric names differ in your podinfo version, use the names printed above.

In [ ]:
%%bash
sleep 60
echo "Target health for podinfo:"
sh labs/12/q.sh 'up{namespace="lab-12"}'
echo; echo "Request rate by status code (req/s):"
sh labs/12/q.sh 'sum by (status) (rate(http_requests_total{namespace="lab-12"}[2m]))'
echo; echo "Error ratio (5xx / all):"
sh labs/12/q.sh 'sum(rate(http_requests_total{namespace="lab-12",status=~"5.."}[2m])) / sum(rate(http_requests_total{namespace="lab-12"}[2m]))'
echo; echo "p95 latency in seconds (histogram):"
sh labs/12/q.sh 'histogram_quantile(0.95, sum by (le) (rate(http_request_duration_seconds_bucket{namespace="lab-12"}[2m])))'

**Expected:** `up` = 1 for two podinfo pods. About one third of requests are `500`. p95 latency close to 1 second (because of `/delay/1`).

### Troubleshooting Prometheus

| Symptom | Cause | Fix |
|---|---|---|
| ServiceMonitor target not in *Targets* | Prometheus does not select it (label selector) | Check `serviceMonitorSelector` on the `Prometheus` object; in default installs add label `release: <helm-release>` |
| Target shown but no endpoints | ServiceMonitor `port` name ≠ Service port name, or Service labels don't match | Compare names; `kubectl get endpointslices` |
| Target DOWN `connection refused` | Wrong port / app not listening on metrics port | `kubectl exec` and `wget localhost:PORT/metrics` |
| Prometheus OOMKilled | Too many series (high **cardinality**, e.g. user ID in a label) | Remove high-cardinality labels; raise memory; `topk(10, count by (__name__)({__name__=~".+"}))` |
| `rate()` returns nothing | Range too short vs scrape interval | Range ≥ 4 × scrape interval |

### Lab vs production

| Lab | Production |
|---|---|
| 1 replica, 1d retention, no PVC | 2 replicas (HA), PVC, 15d local + Thanos/Mimir for long term |
| Select all ServiceMonitors | Controlled selectors per team; limits on samples per scrape |
| No resource tuning | Size by active series; monitor Prometheus itself |

### Check yourself
1. Why must you use `rate()` on `http_requests_total`?
2. What does kube-state-metrics provide that cAdvisor does not?
3. Your ServiceMonitor exists but the target is missing. Name two things to check.
4. What is "high cardinality" and why is it dangerous?

<details><summary>Answer</summary>

1. It is a counter that only increases (and resets on restart). `rate()` turns it into per-second speed and handles resets.
2. Object **state**: pod phase, restart counts, desired vs available replicas, requests/limits, node conditions.
3. The Prometheus `serviceMonitorSelector` labels; the Service labels and the port **name** in the ServiceMonitor; the namespaceSelector.
4. Too many unique label combinations (e.g. a label per user or request ID). Each combination is a new series, so memory and disk explode and Prometheus can crash.
</details>

---
## 4. Grafana

### What is it?
**Grafana** is a web UI for dashboards. It does not store metrics itself. It queries **data sources**: Prometheus, Thanos, Loki, Tempo, OCI Monitoring, and more.

### Lab: explore built-in dashboards
1. Open **http://localhost:3000**. Login `admin` / `learn-admin`.
2. Go to **Dashboards**. kube-prometheus-stack ships many dashboards. Open:
   - *Kubernetes / Compute Resources / Namespace (Pods)* → choose namespace `lab-12`.
   - *Node Exporter / Nodes*.
3. **Import a community dashboard:** Dashboards → New → Import → ID **1860** (Node Exporter Full) → choose the Prometheus data source → Import.

### Lab: build your own RED panel for podinfo
1. Dashboards → New → New dashboard → Add visualization → data source **Prometheus**.
2. Query A (rate): `sum by (status) (rate(http_requests_total{namespace="lab-12"}[2m]))` → legend `{{status}}`.
3. Add a second panel, **Errors %**: `100 * sum(rate(http_requests_total{namespace="lab-12",status=~"5.."}[2m])) / sum(rate(http_requests_total{namespace="lab-12"}[2m]))`, unit *Percent (0-100)*, threshold red at 5.
4. Third panel, **p95 latency**: `histogram_quantile(0.95, sum by (le) (rate(http_request_duration_seconds_bucket{namespace="lab-12"}[2m])))`, unit *seconds*.
5. Save as "podinfo RED".

### Dashboards as code
In production, dashboards are stored in Git and loaded automatically. With kube-prometheus-stack, a ConfigMap with label `grafana_dashboard: "1"` is picked up by the Grafana sidecar. Example (small dashboard JSON):

In [ ]:
%%writefile labs/12/dashboard-cm.yaml
apiVersion: v1
kind: ConfigMap
metadata:
  name: podinfo-dashboard
  namespace: monitoring
  labels:
    grafana_dashboard: "1"          # The Grafana sidecar loads every ConfigMap with this label
data:
  podinfo.json: |
    {
      "title": "podinfo (from Git)",
      "uid": "podinfo-git",
      "schemaVersion": 39,
      "time": {"from": "now-30m", "to": "now"},
      "panels": [
        {
          "type": "timeseries",
          "title": "Requests per second by status",
          "gridPos": {"x": 0, "y": 0, "w": 12, "h": 8},
          "targets": [
            {"expr": "sum by (status) (rate(http_requests_total{namespace=\"lab-12\"}[2m]))",
             "legendFormat": "{{status}}"}
          ]
        }
      ]
    }

In [ ]:
%%bash
kubectl apply -f labs/12/dashboard-cm.yaml
sleep 15
curl -s -u admin:learn-admin "localhost:3000/api/search?query=podinfo" | python3 -m json.tool | grep -E '"title"|"uid"'

**Expected:** `"title": "podinfo (from Git)"`. The dashboard appears in Grafana without clicking anything.

### Troubleshooting Grafana
| Symptom | Cause | Fix |
|---|---|---|
| Panel "No data" | Wrong data source, wrong query, or time range | Test the query in Prometheus UI first |
| Dashboard ConfigMap not loaded | Missing label, or sidecar watches other namespaces | Check label `grafana_dashboard: "1"`, sidecar logs |
| Lost dashboards after restart | No persistence and created in UI | Keep dashboards as code (ConfigMaps / Git) |

### Check yourself
1. Does Grafana store metrics?
2. How do you make dashboards survive a Grafana reinstall?

<details><summary>Answer</summary>

1. No. It queries data sources like Prometheus or Thanos.
2. Store them as JSON in Git and load them with ConfigMaps (sidecar) or provisioning files — or use a database with persistence.
</details>

---
## 5. Alertmanager and alert rules

### What is it?
- **Alert rules** live in Prometheus. A rule is a PromQL expression + a duration (`for:`). If the expression is true for that long, the alert **fires**.
- **Alertmanager** receives firing alerts and decides **what to do**: group them, remove duplicates (e.g. from 2 HA Prometheus replicas), silence them, and route them to receivers (Slack, Teams, email, PagerDuty, webhook).

```
Prometheus rule engine ──firing alerts──▶ Alertmanager
                                            │ group_by: [alertname, namespace]
                                            │ route by labels (severity, team)
                                            ├─ severity=critical ──▶ on-call phone
                                            ├─ severity=warning  ──▶ Teams channel
                                            └─ silences / inhibition
```

Alert states: **inactive → pending** (expression true, waiting for `for:`) **→ firing**.

### Real production scenario
A disk fills slowly. Without a rule, someone notices when the database stops. With a rule "disk will be full in 4 hours" (`predict_linear`) the team gets a warning during working hours and expands the volume calmly — just like your disk-utilisation alert work.

### Lab: PrometheusRule

In [ ]:
%%writefile labs/12/rules.yaml
apiVersion: monitoring.coreos.com/v1
kind: PrometheusRule
metadata:
  name: lab-12-rules
  namespace: lab-12
spec:
  groups:
    - name: lab-12.apps
      rules:
        - alert: PodCrashLooping
          # More than 2 restarts in 5 minutes for any container in lab-12
          expr: increase(kube_pod_container_status_restarts_total{namespace="lab-12"}[5m]) > 2
          for: 1m                         # Must be true for 1 minute (lab: short; prod: 5-15m)
          labels:
            severity: warning             # Used by Alertmanager routing
          annotations:
            summary: "Pod {{ $labels.pod }} is crash looping"
            description: "Container {{ $labels.container }} restarted {{ $value | humanize }} times in 5m."
            runbook_url: "https://wiki.example.com/runbooks/pod-crashlooping"
        - alert: ContainerMemoryNearLimit
          # working set / memory limit > 90%
          expr: |
            sum by (namespace,pod,container) (container_memory_working_set_bytes{namespace="lab-12",container!=""})
              / sum by (namespace,pod,container) (kube_pod_container_resource_limits{namespace="lab-12",resource="memory"})
              > 0.9
          for: 2m
          labels:
            severity: warning
          annotations:
            summary: "{{ $labels.pod }}/{{ $labels.container }} memory above 90% of limit"
        - alert: HighErrorRate
          # Symptom-based alert: users see errors
          expr: |
            sum(rate(http_requests_total{namespace="lab-12",status=~"5.."}[5m]))
              / sum(rate(http_requests_total{namespace="lab-12"}[5m])) > 0.05
          for: 2m
          labels:
            severity: critical
          annotations:
            summary: "podinfo error rate above 5%"

Apply the rules, and start a crash-looping pod to trigger `PodCrashLooping`. Our podinfo load already has ~33% errors, so `HighErrorRate` will also fire.

In [ ]:
%%bash
kubectl apply -f labs/12/rules.yaml
kubectl -n lab-12 run crashloop --image=busybox:1.36 --restart=Always -- sh -c 'echo boom; sleep 2; exit 1'
echo "Waiting ~4 minutes for restarts + 'for' duration..."
sleep 240
echo "Alerts in Prometheus:"
curl -s localhost:9090/api/v1/alerts | python3 -c '
import json,sys
for a in json.load(sys.stdin)["data"]["alerts"]:
    if a["labels"].get("namespace")=="lab-12" or a["labels"]["alertname"]=="HighErrorRate":
        print(a["state"].ljust(8), a["labels"]["alertname"], a["labels"].get("pod",""))'

**Expected:** `firing PodCrashLooping crashloop` and `firing HighErrorRate`. (If still `pending`, wait one more minute and re-run the query part.)

Now check that Alertmanager received them:

In [ ]:
%%bash
curl -s localhost:9093/api/v2/alerts | python3 -c '
import json,sys
for a in json.load(sys.stdin):
    l=a["labels"]
    if l.get("namespace")=="lab-12" or l["alertname"]=="HighErrorRate":
        print(l["alertname"], l.get("severity"), a["status"]["state"])'

### Routing (Alertmanager configuration)
With kube-prometheus-stack you set the routing in Helm values (`alertmanager.config`) or with namespaced `AlertmanagerConfig` objects. Example values (read; uses a placeholder webhook):

```yaml
alertmanager:
  config:
    route:
      receiver: default                 # fallback receiver
      group_by: [alertname, namespace]  # one notification per alert type per namespace
      group_wait: 30s                   # wait to collect similar alerts
      group_interval: 5m
      repeat_interval: 4h               # re-send if still firing
      routes:
        - matchers: ['severity="critical"']
          receiver: oncall
        - matchers: ['severity="warning"']
          receiver: team-channel
    inhibit_rules:                      # if the node is down, don't also page for every pod on it
      - source_matchers: ['alertname="KubeNodeNotReady"']
        target_matchers: ['severity="warning"']
        equal: [node]
    receivers:
      - name: default
      - name: oncall
        webhook_configs: [{url: "https://oncall.example.com/hook"}]
      - name: team-channel
        msteams_configs: [{webhook_url: "https://example.webhook.office.com/..."}]
```

**Silences:** in the Alertmanager UI (http://localhost:9093) click *New Silence* to mute an alert during planned maintenance (for example the patch window). Try: silence `alertname=PodCrashLooping` for 1 hour.

Delete the crashing pod now:

In [ ]:
%%bash
kubectl -n lab-12 delete pod crashloop --now

### Troubleshooting alerts
| Symptom | Cause | Fix |
|---|---|---|
| Rule not visible in Prometheus *Rules* page | `ruleSelector` doesn't match the PrometheusRule labels | Add the required label (default: `release: <name>`) |
| Alert stays `pending` | `for:` not reached, or flapping | Wait; check expression over time in Graph tab |
| Firing in Prometheus, no notification | Alertmanager route/receiver wrong, or silenced | Alertmanager UI → status and silences; logs |
| Same alert 2x | Not grouped, or two Alertmanagers not clustered | Use HA Alertmanager cluster; same `external_labels` |

### Check yourself
1. What does `for: 5m` do?
2. What is the job of Alertmanager that Prometheus does not do?
3. What is an inhibition rule? Give an example.

<details><summary>Answer</summary>

1. The expression must stay true for 5 minutes before the alert fires. It avoids alerts for short spikes.
2. Grouping, deduplication, silencing, inhibition and routing to receivers.
3. A rule that mutes some alerts when another alert fires. Example: node down → mute all pod alerts on that node.
</details>

---
## 6. Loki — logs, the Prometheus way

### What is it?
**Loki** is a log store from Grafana Labs. It indexes only **labels** (namespace, pod, app), not the full text. This makes it cheap. You search with **LogQL**: first choose streams by labels, then filter text.

### How it works

```
Alloy (collector) ──push──▶ Loki (distributor → ingester → chunks on object storage)
                                             ▲
Grafana (Explore, LogQL) ──query─────────────┘
```

Production Loki stores chunks in object storage (OCI Object Storage via the S3-compatible API, S3, GCS). Lab: local filesystem in **single binary** mode.

### Lab: install Loki (single binary, filesystem)

In [ ]:
%%writefile labs/12/loki-values.yaml
deploymentMode: SingleBinary          # All Loki components in one pod (lab / small installs)
loki:
  auth_enabled: false                 # No multi-tenancy header needed (lab)
  commonConfig:
    replication_factor: 1
  storage:
    type: filesystem                  # Production: s3 / gcs / azure (object storage)
  schemaConfig:
    configs:
      - from: "2024-04-01"
        store: tsdb
        object_store: filesystem
        schema: v13
        index: {prefix: loki_index_, period: 24h}
  limits_config:
    retention_period: 24h
singleBinary:
  replicas: 1
  resources:
    requests: {cpu: 50m, memory: 128Mi}
    limits: {memory: 512Mi}
# Turn off the components used by scalable modes:
read:    {replicas: 0}
write:   {replicas: 0}
backend: {replicas: 0}
gateway: {enabled: false}
chunksCache:  {enabled: false}
resultsCache: {enabled: false}
lokiCanary:   {enabled: false}
test:         {enabled: false}

In [ ]:
%%bash
helm upgrade --install loki grafana/loki -n monitoring -f labs/12/loki-values.yaml --wait --timeout 8m
kubectl -n monitoring get pods -l app.kubernetes.io/name=loki
kubectl -n monitoring get svc loki

### Ship logs with Grafana Alloy
Alloy uses a config language with **components** connected like a pipeline. This config: discover pods → set labels → read logs through the Kubernetes API → push to Loki.

We run Alloy as a **Deployment with 1 replica** because `loki.source.kubernetes` reads logs via the API for the whole cluster. (As a DaemonSet each pod would read all pods → duplicate logs, unless you enable clustering or read node files instead.)

In [ ]:
%%writefile labs/12/alloy-values.yaml
controller:
  type: deployment                    # One collector for the whole (small) cluster
  replicas: 1
alloy:
  resources:
    requests: {cpu: 20m, memory: 64Mi}
  configMap:
    content: |
      // 1. Find all pods using the Kubernetes API
      discovery.kubernetes "pods" {
        role = "pod"
      }
      // 2. Turn Kubernetes metadata into Loki labels (keep labels FEW = low cardinality)
      discovery.relabel "pods" {
        targets = discovery.kubernetes.pods.targets
        rule {
          source_labels = ["__meta_kubernetes_namespace"]
          target_label  = "namespace"
        }
        rule {
          source_labels = ["__meta_kubernetes_pod_name"]
          target_label  = "pod"
        }
        rule {
          source_labels = ["__meta_kubernetes_pod_container_name"]
          target_label  = "container"
        }
        rule {
          source_labels = ["__meta_kubernetes_pod_label_app"]
          target_label  = "app"
        }
      }
      // 3. Tail container logs through the API server
      loki.source.kubernetes "pods" {
        targets    = discovery.relabel.pods.output
        forward_to = [loki.write.default.receiver]
      }
      // 4. Send to Loki
      loki.write "default" {
        endpoint {
          url = "http://loki.monitoring.svc.cluster.local:3100/loki/api/v1/push"
        }
      }

In [ ]:
%%bash
helm upgrade --install alloy grafana/alloy -n monitoring -f labs/12/alloy-values.yaml --wait --timeout 5m
kubectl -n monitoring get pods -l app.kubernetes.io/name=alloy

Add Loki as a Grafana data source using a ConfigMap. The Grafana sidecar loads ConfigMaps labelled `grafana_datasource: "1"`.

In [ ]:
%%writefile labs/12/loki-datasource.yaml
apiVersion: v1
kind: ConfigMap
metadata:
  name: loki-datasource
  namespace: monitoring
  labels:
    grafana_datasource: "1"          # Grafana sidecar picks this up
data:
  loki.yaml: |
    apiVersion: 1
    datasources:
      - name: Loki
        type: loki
        uid: loki
        access: proxy
        url: http://loki.monitoring.svc.cluster.local:3100

In [ ]:
%%bash
kubectl apply -f labs/12/loki-datasource.yaml
sleep 20
curl -s -u admin:learn-admin localhost:3000/api/datasources | python3 -c 'import json,sys; [print(d["name"], d["type"]) for d in json.load(sys.stdin)]'

If `Loki` is not listed after a minute, restart Grafana so it reloads data sources: `kubectl -n monitoring rollout restart deploy/kps-grafana` (then restart its port-forward by re-running the port-forward cell).

### LogQL basics
Run queries against the Loki API through a temporary port-forward:

| Query | Meaning |
|---|---|
| `{namespace="lab-12"}` | All logs in namespace lab-12 |
| `{namespace="lab-12", app="logger"} \|= "error"` | Lines containing "error" |
| `{app="logger"} \| logfmt \| level="error"` | Parse key=value, filter by field |
| `sum by (pod) (count_over_time({namespace="lab-12"} \|= "error" [5m]))` | Error lines per pod in 5 min (a metric from logs) |

In [ ]:
%%bash
nohup kubectl -n monitoring port-forward svc/loki 3100:3100 > labs/12/pf-loki.log 2>&1 &
echo $! > labs/12/pf-loki.pid
sleep 4
lq() {  # instant/range query helper
  curl -s -G localhost:3100/loki/api/v1/query_range --data-urlencode "query=$1" --data-urlencode "limit=5" \
  | python3 -c '
import json,sys
d=json.load(sys.stdin)["data"]
for r in d["result"][:5]:
    if d["resultType"]=="streams":
        for ts,line in r["values"][:3]: print(r["stream"].get("pod"), "|", line[:110])
    else:
        print(r["metric"], "=>", r["values"][-1][1])'
}
echo "1) Error lines from logger:"
lq '{namespace="lab-12", app="logger"} |= "error"'
echo; echo "2) Parsed with logfmt, only level=error:"
lq '{namespace="lab-12", app="logger"} | logfmt | level="error"'
echo; echo "3) Error count per pod over 5m (metric from logs):"
lq 'sum by (pod) (count_over_time({namespace="lab-12"} |= "error" [5m]))'

**Expected:** lines like `logger | 2026-...Z level=error msg="payment timeout" order=35`, and a count per pod. In Grafana: **Explore → Loki** → run the same queries.

### Troubleshooting Loki
| Symptom | Cause | Fix |
|---|---|---|
| No logs in Loki | Collector not running / wrong push URL | `kubectl -n monitoring logs deploy/alloy`, check URL |
| `429 Too Many Requests` / rate limited | Ingestion limits exceeded | Raise `limits_config` ingestion rate, reduce log volume |
| `entry out of order` / `too far behind` | Clock skew or replayed old logs | Sync clocks (NTP); tune `reject_old_samples` |
| Queries slow, Loki OOM | Too many labels (e.g. request_id as label) | Keep labels low-cardinality; put IDs in the line, filter with `\|=` |

### Lab vs production
| Lab | Production |
|---|---|
| SingleBinary, filesystem, 24h | Simple-scalable or distributed mode, object storage (OCI Object Storage S3 API), 30–90 day retention |
| 1 Alloy Deployment via API | Alloy/Fluent Bit DaemonSet reading node files (scales with nodes) |
| No auth | Multi-tenant (`X-Scope-OrgID`), auth via gateway |

### Check yourself
1. What does Loki index? Why is it cheaper than full-text search engines?
2. Why should `request_id` not be a Loki label?
3. Write a LogQL query for "lines containing timeout in namespace payments".

<details><summary>Answer</summary>

1. Only labels. The log text is stored compressed in chunks and filtered at query time, so the index stays small.
2. High cardinality: one stream per request → huge index, slow and expensive. Keep it in the log line.
3. `{namespace="payments"} |= "timeout"`
</details>

---
## 7. OpenTelemetry and distributed tracing

### What is it?
- **Distributed tracing** follows one request across many services. A **trace** = the whole request. A **span** = one step (e.g. "call payment API", 800 ms). Spans have a parent, so you see a tree.
- **OpenTelemetry (OTel)** is the open standard (CNCF) for creating and sending traces, metrics and logs. Apps use OTel SDKs (or **auto-instrumentation**). Data is sent with the **OTLP** protocol (gRPC port 4317, HTTP port 4318).
- The **OpenTelemetry Collector** receives, processes (batch, filter, add Kubernetes labels) and exports data to backends.

```
trace_id=abc123
checkout  [==================== 1200 ms ====================]
  ├─ cart-service   [== 80 ms ==]
  ├─ payment        [============== 900 ms ==============]
  │    └─ bank-api      [=========== 850 ms ===========]   ← slow part!
  └─ email (async)                                  [= 50 ms =]

app (OTel SDK) ──OTLP──▶ OTel Collector ──▶ Tempo / Jaeger (traces)
                                       ├──▶ Prometheus (metrics)
                                       └──▶ Loki (logs)
```

- **Tempo** (Grafana) and **Jaeger** are trace backends. Grafana can jump from a log line with `trace_id` to the trace in Tempo.
- **Auto-instrumentation:** the **OpenTelemetry Operator** can inject agents into Java, Python, Node.js, .NET and Go pods using an annotation like `instrumentation.opentelemetry.io/inject-java: "true"`. No code change.

### Real production scenario
Checkout is slow (p95 4 s) but every service dashboard looks "OK". A trace shows that payment calls a fraud-check API **three times in sequence**. The fix (parallel calls) cuts p95 to 1.2 s. Metrics alone could not show this.

### Lab: OpenTelemetry Collector + test traces

In [ ]:
%%writefile labs/12/otel-values.yaml
mode: deployment                      # One collector Deployment (alternatives: daemonset, statefulset)
fullnameOverride: otel-collector      # Service name: otel-collector
image:
  repository: otel/opentelemetry-collector-k8s   # Required by the chart: choose a distribution
resources:
  requests: {cpu: 20m, memory: 64Mi}
  limits: {memory: 256Mi}
config:
  receivers:
    otlp:
      protocols:
        grpc: {endpoint: "0.0.0.0:4317"}   # OTLP gRPC
        http: {endpoint: "0.0.0.0:4318"}   # OTLP HTTP
  processors:
    batch: {}                         # Send data in batches (always use in production)
  exporters:
    debug:
      verbosity: detailed             # Print received data to the collector log (lab only)
  service:
    pipelines:
      traces:
        receivers: [otlp]
        processors: [batch]
        exporters: [debug]            # Production: otlp exporter to Tempo / Jaeger / a vendor

In [ ]:
%%bash
helm upgrade --install otel open-telemetry/opentelemetry-collector -n monitoring -f labs/12/otel-values.yaml --wait --timeout 5m
kubectl -n monitoring get pods -l app.kubernetes.io/instance=otel
kubectl -n monitoring get svc otel-collector

Now send 3 test traces with **telemetrygen** (a test tool from the OTel project). It acts like an instrumented app.

In [ ]:
%%bash
kubectl -n lab-12 run telemetrygen --rm -i --restart=Never \
  --image=ghcr.io/open-telemetry/opentelemetry-collector-contrib/telemetrygen:v0.120.0 -- \
  traces --otlp-endpoint=otel-collector.monitoring.svc.cluster.local:4317 --otlp-insecure \
  --traces=3 --service=checkout-demo 2>&1 | tail -3
sleep 5
kubectl -n monitoring logs deploy/otel-collector --tail=200 | grep -E 'Trace ID|Name|service.name' | head -12

**Expected:** collector logs show `service.name: Str(checkout-demo)`, `Trace ID`, span names like `lets-go` and `okey-dokey-0`. In production the exporter sends these to Tempo/Jaeger and you view the tree in Grafana.

### Troubleshooting OTel
| Symptom | Cause | Fix |
|---|---|---|
| App sends nothing | SDK endpoint wrong (`OTEL_EXPORTER_OTLP_ENDPOINT`) | Point to the collector Service, right port (4317 gRPC / 4318 HTTP) |
| `connection refused` / TLS errors | Insecure vs TLS mismatch | Match `--otlp-insecure` / TLS settings |
| Traces broken into pieces | Context not propagated (missing `traceparent` header) | Use OTel instrumentation for HTTP clients; same propagator everywhere |
| Collector OOM | No `memory_limiter` processor, big bursts | Add `memory_limiter` + `batch` processors |

### Check yourself
1. Trace vs span?
2. What protocol and ports does OTLP use?
3. What does the OTel Collector add compared to sending directly to Tempo?

<details><summary>Answer</summary>

1. A trace is one end-to-end request; a span is one operation inside it with start time, duration and parent.
2. OTLP over gRPC (4317) or HTTP (4318).
3. Vendor neutrality (change backend without changing apps), batching, retries, filtering, sampling, adding Kubernetes metadata, sending to many backends.
</details>

---
## 8. What to monitor: RED, USE, golden signals

### Application monitoring — **RED** (for request-driven services)
| Letter | Meaning | PromQL example (podinfo) |
|---|---|---|
| **R**ate | Requests per second | `sum(rate(http_requests_total[5m]))` |
| **E**rrors | Failed requests per second (or %) | `sum(rate(http_requests_total{status=~"5.."}[5m]))` |
| **D**uration | Latency (p50/p95/p99) | `histogram_quantile(0.95, sum by (le)(rate(http_request_duration_seconds_bucket[5m])))` |

### Infrastructure monitoring — **USE** (for resources: CPU, memory, disk, network)
| Letter | Meaning | Example |
|---|---|---|
| **U**tilization | % busy | node CPU %, disk used % |
| **S**aturation | Extra work waiting | CPU load / run queue, **CPU throttling**, memory pressure, disk I/O wait |
| **E**rrors | Error events | network errors, disk errors, OOM kills |

Kubernetes-specific saturation example (CPU throttling — pod hits its CPU limit):
```
sum by (pod) (rate(container_cpu_cfs_throttled_periods_total[5m]))
  / sum by (pod) (rate(container_cpu_cfs_periods_total[5m]))
```

### The four golden signals (Google SRE)
**Latency, Traffic, Errors, Saturation.** RED covers the first three; saturation comes from USE.

### Cluster monitoring checklist
- Nodes: Ready, CPU/memory/disk, disk pressure, `node_filesystem_avail_bytes`.
- Workloads: restarts, OOMKilled, Pending pods, Deployments with `available < desired`.
- Control plane (self-managed): API server latency and errors, etcd leader changes and DB size, scheduler pending pods.
- Certificates: days until expiry.
- Capacity: requests vs allocatable per node (can we schedule more?).

---
## 9. Alerting strategy

Good alerts wake people up **only** for real user-facing problems.

| Principle | Meaning |
|---|---|
| **Alert on symptoms, not causes** | Page on "error rate > 5%" or "p99 > 2 s", not on "CPU 90%" (high CPU may be fine) |
| **Every page is actionable** | If nobody needs to act now, it is a ticket or a dashboard, not a page |
| **Use `for:`** | Avoid flapping on short spikes |
| **Severity levels** | `critical` = page now (users affected); `warning` = fix in working hours; `info` = dashboard only |
| **Runbook for every alert** | `runbook_url` annotation: what it means, how to check, how to fix |
| **SLO-based alerts** | Alert on error-budget **burn rate** (e.g. burning 14x faster than allowed) |
| **Review alerts monthly** | Delete alerts nobody acted on; tune noisy ones |

**Alert fatigue** = so many alerts that people ignore them — and miss the real one. A team that gets 200 alerts per night has no alerting.

Example of good predictive warning (disk will be full within 4 hours):
```
predict_linear(node_filesystem_avail_bytes{mountpoint="/"}[6h], 4*3600) < 0
```

### Production dashboards — a good layout
1. **Overview** (one screen): golden signals per service, SLO status, active alerts.
2. **Service dashboard**: RED by endpoint and version (useful for canary from notebook 11), pods, restarts, resources.
3. **Cluster/capacity**: nodes (USE), requests vs allocatable, Pending pods, top consumers.
4. **Dependencies**: databases, queues, external APIs.

Rules: top-left = most important; same time range and colours everywhere; link panels to logs (Loki) and traces (Tempo); dashboards stored as code.

### Check yourself
1. Why is "CPU > 90%" usually a bad page alert?
2. Which RED signal does a canary analysis usually check first?
3. What is alert fatigue?

<details><summary>Answer</summary>

1. High CPU alone does not mean users suffer. It may be normal batch work. Alert on latency/errors instead; use CPU for dashboards and capacity planning.
2. Errors (error rate of the canary vs stable), then duration.
3. Too many non-actionable alerts, so people start ignoring them and miss real incidents.
</details>

---
## 10. Interview and real-world questions

1. What is the difference between monitoring and observability?
2. How would you monitor 10 OKE clusters centrally?
3. Prometheus is using 12 GB of memory and growing. What do you do?
4. A user says "the site was slow at 14:00". How do you investigate with metrics, logs and traces?
5. How do you avoid duplicate alerts from two HA Prometheus replicas?

<details><summary>Answer</summary>

1. Monitoring checks known problems with predefined metrics and alerts. Observability lets you ask new questions about unknown problems using rich logs, metrics and traces.
2. A Prometheus (or Alloy agent) in each cluster with a `cluster` external label, `remote_write` or Thanos sidecar to a central Thanos/Mimir with object storage; central Grafana and Alertmanager; same rules everywhere via GitOps. (This is close to your current design.)
3. Find high-cardinality metrics (`topk(10, count by (__name__)({__name__=~".+"}))`, TSDB status page), drop or relabel bad labels, reduce scrape targets/intervals, shard Prometheus, move long-term to Thanos.
4. Metrics: RED dashboard at 14:00 — which service, which endpoint, all users or some? Traces: slow traces from that time → which span is slow. Logs: filter by trace_id / service / time for errors or timeouts. Check changes at that time (deploys, CRs).
5. Both replicas send to the same Alertmanager cluster; Alertmanager deduplicates identical alerts (same labels). Use the same rules and `external_labels` except a replica label that is dropped for alerts.
</details>

---
## 11. Practical assignment

**Scenario:** The `orders` team asks for monitoring of their new service before go-live.

Deploy a second podinfo called `orders` in `lab-12` and deliver:
1. Prometheus scrapes it (target UP).
2. A Grafana dashboard **from a ConfigMap** with RED panels for `orders` only.
3. An alert `OrdersHighLatency`: p95 > 0.5 s for 2 minutes, severity `warning`, with a `runbook_url`.
4. A LogQL query that shows only `orders` error lines.

**Hints:** copy `podinfo.yaml`, rename to `orders` (labels too); generate load to `/delay/1`; filter queries with `pod=~"orders-.*"`.

<details><summary>Solution</summary>

1. Copy the Deployment/Service/ServiceMonitor with `name: orders`, labels `app: orders`, ServiceMonitor `matchLabels: {app: orders}`. Check `up{namespace="lab-12",pod=~"orders-.*"}` = 1.
2. ConfigMap with label `grafana_dashboard: "1"` and three panels using `{namespace="lab-12", pod=~"orders-.*"}` in the queries (rate, error %, p95).
3. Rule:
```yaml
- alert: OrdersHighLatency
  expr: histogram_quantile(0.95, sum by (le) (rate(http_request_duration_seconds_bucket{namespace="lab-12",pod=~"orders-.*"}[5m]))) > 0.5
  for: 2m
  labels: {severity: warning}
  annotations:
    summary: "orders p95 latency above 500ms"
    runbook_url: "https://wiki.example.com/runbooks/orders-latency"
```
Generate load: `kubectl -n lab-12 run orders-load --image=curlimages/curl:8.10.1 --command -- sh -c 'while true; do curl -s http://orders:9898/delay/1; done'`.
4. `{namespace="lab-12", app="orders"} |= "error"` (or `| json | level="error"` — podinfo logs JSON).
</details>

---
## 12. Break it challenge

Apply this ServiceMonitor. The target never appears in Prometheus. Find **two** mistakes.

In [ ]:
%%writefile labs/12/break-it.yaml
apiVersion: monitoring.coreos.com/v1
kind: ServiceMonitor
metadata:
  name: podinfo-broken
  namespace: lab-12
spec:
  selector:
    matchLabels: {app: pod-info}
  namespaceSelector:
    matchNames: [lab-12]
  endpoints:
    - port: "9797"
      path: /metrics

In [ ]:
%%bash
kubectl apply -f labs/12/break-it.yaml
sleep 45
sh labs/12/q.sh 'count by (job) (up{namespace="lab-12"})'

<details><summary>Solution</summary>

1. `matchLabels: {app: pod-info}` — the Service label is `app: podinfo`. No Service matches, so no target.
2. `port: "9797"` — in a ServiceMonitor, `port` is the **name** of the Service port (`http-metrics`), not the number. (Use `targetPort: 9797` if you really want a number.)

Debug commands: `kubectl get svc -n lab-12 --show-labels`, `kubectl get svc podinfo -n lab-12 -o yaml | grep -A3 ports`, Prometheus UI → *Status → Service Discovery* (shows dropped targets).

Fix: `kubectl delete -f labs/12/break-it.yaml` (the working `podinfo` ServiceMonitor already exists).
</details>

---
## 13. Cleanup

Stops port-forwards and deletes `lab-12`.

**Keep the monitoring stack?** Notebook 13 (troubleshooting) is easier with metrics-server and Prometheus. If your Mac has enough memory, keep `monitoring` and skip the `helm uninstall` lines. metrics-server is kept (HPA and `kubectl top` need it).

In [ ]:
%%bash
for f in labs/12/pf-*.pid; do [ -f "$f" ] && kill $(cat "$f") 2>/dev/null; rm -f "$f"; done
kubectl delete namespace lab-12 --ignore-not-found
kubectl -n monitoring delete configmap podinfo-dashboard --ignore-not-found
# Remove the monitoring tools (skip these lines to keep them):
helm uninstall otel alloy loki -n monitoring 2>/dev/null
helm uninstall kps -n monitoring 2>/dev/null
kubectl -n monitoring delete pvc --all --ignore-not-found
kubectl delete namespace monitoring --ignore-not-found
# kube-prometheus-stack leaves its CRDs (ServiceMonitor, PrometheusRule, ...). Remove only if you will not reinstall:
# kubectl get crd -o name | grep monitoring.coreos.com | xargs kubectl delete
echo "Cleanup done."

**Next notebook:** [13-Troubleshooting-and-Incidents.ipynb](13-Troubleshooting-and-Incidents.ipynb) — investigate realistic production incidents and fix them.